# Laboratorio 2 — Auditoría exploratoria de un conjunto de datos

Un conjunto de datos de mantenimiento industrial llega sin documentación: 4237 registros de
sensores de máquinas, con la variable `fallo` como objetivo. Antes de modelar nada hay que
saber **qué tiene adentro**.

El laboratorio recorre la auditoría completa: estructura, códigos de error, medidas de posición,
la forma de cada distribución, el diagrama de caja y lo que esconde, los atípicos, las
relaciones entre variables y los faltantes. Cada paso produce evidencia, y la evidencia decide
cómo se modela después.

**El entregable es el informe de hallazgos de la sección final**: por cada hallazgo, evidencia,
criterio, decisión y justificación. Lo que no se complete durante la clase se termina fuera de
ella; el informe se entrega igual.

Las celdas que contienen `raise NotImplementedError` deben completarse.

## 0. Preparación

El conjunto de datos se distribuye como archivo: `data/mantenimiento_industrial.csv`. Lo produce
el script `src/generar_dataset_sucio.py` con semilla fija, de modo que todos trabajan sobre
exactamente las mismas filas.

La celda siguiente lo busca en las ubicaciones habituales. En Google Colab, si no lo encuentra,
abre el diálogo de carga de archivos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

In [ ]:
from pathlib import Path

NOMBRE = "mantenimiento_industrial.csv"
CANDIDATAS = [Path(NOMBRE), Path("data") / NOMBRE, Path("../../data") / NOMBRE]


def cargar_datos() -> pd.DataFrame:
    """Carga el CSV del curso desde la primera ubicación donde exista.

    En Colab, si no está en ninguna, pide subirlo.
    """
    for ruta in CANDIDATAS:
        if ruta.exists():
            print(f"Cargado desde {ruta}")
            return pd.read_csv(ruta)
    try:
        from google.colab import files  # type: ignore
    except ImportError as exc:
        raise FileNotFoundError(
            f"No se encontró {NOMBRE}. Copiarlo junto al notebook."
        ) from exc
    files.upload()
    return pd.read_csv(NOMBRE)


df = cargar_datos()
df.shape

### Para analizar

Antes de calcular nada, corresponde mirar qué llegó: cuántas filas y columnas, de qué tipo es
cada una y qué valores toma.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

Tres cosas de esa salida merecen una anotación inmediata:

1. `temperatura_c` tiene máximo 999,0. No es una temperatura.
2. `presion_bar` tiene mínimo −1,0. Una presión negativa en este proceso no es **posible**.
3. `horas_operacion` llega a valores seis órdenes por encima de `vibracion_mm_s`. Cualquier
   método basado en distancias va a estar dominado por la primera.

## 1. Estructura: cuántas observaciones distintas hay realmente

`id_registro` es único por construcción, así que `df.duplicated()` devuelve cero aunque haya
filas repetidas. La comparación tiene que hacerse **sin** el identificador.

In [ ]:
def contar_duplicados_reales(df: pd.DataFrame, col_id: str) -> int:
    """Cuenta filas duplicadas ignorando la columna identificadora.

    Parámetros
    ----------
    df : DataFrame con los datos crudos.
    col_id : nombre de la columna identificadora, que debe excluirse de la comparación.

    Devuelve
    --------
    int : cantidad de filas que son repetición de otra fila anterior.
    """
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN
n_dup = contar_duplicados_reales(df, "id_registro")
print(f"Duplicados reales: {n_dup}")
print(f"Duplicados sin excluir el identificador: {df.duplicated().sum()}")
assert n_dup == 37, f"Se esperaban 37, se obtuvieron {n_dup}"
print("Correcto")

### Para analizar

1. ¿Por qué `df.duplicated()` da 0 y `contar_duplicados_reales` da 37?
2. ¿Corresponde eliminar esas 37 filas en esta etapa? ¿Qué haría falta saber para decidirlo?

**Respuesta:**

1.
2.

## 2. Códigos numéricos de error

Un código de error se reconoce por **dos** señales a la vez: se repite exacto muchas veces, y
está lejos del grueso de los datos. Cualquiera de las dos por separado produce falsos positivos:
un valor entero frecuente no es sospechoso, y un valor lejano aislado es un atípico legítimo.

In [ ]:
def detectar_codigos_error(serie: pd.Series, min_repeticiones: int = 10,
                           k: float = 3.0) -> list[float]:
    """Detecta valores sospechosos de ser códigos numéricos de error.

    Criterio: el valor cumple LAS DOS condiciones a la vez —
      (a) se repite exacto al menos `min_repeticiones` veces, y
      (b) cae fuera de [Q1 - k*IQR, Q3 + k*IQR].

    Parámetros
    ----------
    serie : pd.Series numérica, puede contener NaN.
    min_repeticiones : cuántas veces debe repetirse un valor para resultar sospechoso.
    k : amplitud de la valla que define "lejos del grueso".

    Devuelve
    --------
    list[float] : los valores sospechosos.
    """
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN
continuas = ["temperatura_c", "vibracion_mm_s", "presion_bar", "consumo_kwh"]
for col in continuas:
    print(f"{col:20s} -> {detectar_codigos_error(df[col])}")

assert detectar_codigos_error(df["temperatura_c"]) == [999.0]
assert detectar_codigos_error(df["presion_bar"]) == [-1.0]
assert detectar_codigos_error(df["vibracion_mm_s"]) == []
print("\nCorrecto: exactamente dos códigos de error, sin falsos positivos")

Detectados los códigos, se los reemplaza por faltantes. **El resto del laboratorio trabaja
sobre `limpio`**, no sobre `df`.

In [ ]:
limpio = df.replace({"temperatura_c": {999.0: np.nan}, "presion_bar": {-1.0: np.nan}})
print("Faltantes antes y después de reconocer los códigos:")
print(pd.DataFrame({"df": df.isna().sum(), "limpio": limpio.isna().sum()}).query("limpio > 0"))

## 3. Medidas de posición: media y mediana

Corresponde implementar las dos desde cero, con operaciones elementales de NumPy. La mediana
exige ordenar y distinguir el caso par del impar; ese detalle es justamente el que la vuelve
robusta.

In [ ]:
def media(x: np.ndarray) -> float:
    """Media aritmética de un arreglo sin faltantes.

    Parámetros
    ----------
    x : np.ndarray de forma (n,), sin NaN, con n >= 1.

    Devuelve
    --------
    float

    No usar np.mean: la suma y la división bastan.
    """
    raise NotImplementedError()

In [ ]:
def mediana(x: np.ndarray) -> float:
    """Mediana de un arreglo sin faltantes.

    Parámetros
    ----------
    x : np.ndarray de forma (n,), sin NaN, con n >= 1.

    Devuelve
    --------
    float : el valor central si n es impar; el promedio de los dos centrales si es par.

    No usar np.median: ordenar con np.sort y separar los dos casos.
    """
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN
impar = np.array([3.0, 1.0, 2.0])
par = np.array([4.0, 1.0, 3.0, 2.0])
assert np.isclose(mediana(impar), 2.0) and np.isclose(mediana(par), 2.5), "casos de borde"

for col in ["temperatura_c", "vibracion_mm_s", "consumo_kwh"]:
    x = limpio[col].dropna().to_numpy()
    assert np.isclose(media(x), np.mean(x)), f"media mal en {col}"
    assert np.isclose(mediana(x), np.median(x)), f"mediana mal en {col}"
print("Correcto: coinciden con np.mean y np.median en las tres variables")

Comparar las dos medidas es un diagnóstico de asimetría que sale gratis: si
$\bar{x} \gg \tilde{x}$ hay cola derecha, y si $\bar{x} \ll \tilde{x}$, cola izquierda.

In [ ]:
def tabla_posicion(df: pd.DataFrame, columnas: list[str]) -> pd.DataFrame:
    """Compara media y mediana de varias variables numéricas.

    Parámetros
    ----------
    df : DataFrame de entrada (puede contener NaN).
    columnas : nombres de las variables numéricas a resumir.

    Devuelve
    --------
    pd.DataFrame indexado por `columnas`, con las columnas
    ["media", "mediana", "diferencia_relativa"], donde
    diferencia_relativa = (media - mediana) / mediana.
    Ordenado por el valor absoluto de esa diferencia, de mayor a menor.
    Los faltantes se descartan variable por variable.
    """
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN
numericas = ["temperatura_c", "vibracion_mm_s", "presion_bar", "consumo_kwh",
             "horas_operacion", "antiguedad_meses", "n_mantenimientos_previos"]
# n_mantenimientos_previos es un conteo entero de rango muy corto: la comparación
# media/mediana no dice nada sobre su forma. Se excluye de esta tabla.
posicion = tabla_posicion(limpio, [c for c in numericas if c != "n_mantenimientos_previos"])
print(posicion.round(3))

assert list(posicion.columns) == ["media", "mediana", "diferencia_relativa"]
assert posicion["diferencia_relativa"].abs().is_monotonic_decreasing, "falta ordenar"
assert posicion.index[0] == "horas_operacion", \
    "la variable con mayor divergencia debería ser horas_operacion"
print("\nCorrecto")

### Para analizar

1. En `vibracion_mm_s` la media supera a la mediana. ¿Qué forma tiene entonces la distribución?
2. `horas_operacion` encabeza la tabla por un margen enorme. Eso no es asimetría natural: es el
   rastro de un defecto. ¿Cuál puede ser?
3. Para informar «el valor típico» de `vibracion_mm_s`, ¿qué medida corresponde y por qué?

**Respuesta:**

1.
2.
3.

## 4. Variables categóricas: gráficos de barras

Sobre una categórica no hay media ni mediana: hay **frecuencias**. El gráfico de barras es su
representación natural, y muestra de inmediato dos problemas frecuentes: la cardinalidad alta y
la codificación inconsistente.

In [ ]:
def tabla_frecuencias(serie: pd.Series) -> pd.DataFrame:
    """Tabla de frecuencias de una variable categórica.

    Parámetros
    ----------
    serie : pd.Series categórica o de texto, puede contener NaN.

    Devuelve
    --------
    pd.DataFrame indexado por los niveles, con columnas ["n", "porcentaje"],
    ordenado por "n" de mayor a menor. El porcentaje se calcula sobre los no faltantes.
    """
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN
frec = tabla_frecuencias(df["turno"])
print(frec.round(1))
assert list(frec.columns) == ["n", "porcentaje"]
assert np.isclose(frec["porcentaje"].sum(), 100.0), "los porcentajes deben sumar 100"
assert frec["n"].is_monotonic_decreasing, "falta ordenar por frecuencia"
print("\nCorrecto")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["planta", "turno", "modelo_equipo"]):
    frec = tabla_frecuencias(df[col])
    ax.bar(frec.index.astype(str), frec["n"], color="#19326E")
    ax.set_title(f"{col} — {df[col].nunique()} niveles", fontweight="bold")
    ax.set_ylabel("Cantidad de registros")
    ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

print("Cardinalidad de cada categórica:")
for col in ["planta", "turno", "modelo_equipo", "id_maquina"]:
    print(f"  {col:16s} {df[col].nunique():4d} niveles")

### DECIDE

El gráfico de `modelo_equipo` muestra más barras de las que hay modelos de equipo.

1. Inspeccionar los niveles y explicar qué ocurrió. ¿Cuántos modelos distintos hay realmente?
2. `id_maquina` tiene 180 niveles. Si se codificara con *one-hot encoding*, ¿cuántas columnas
   agregaría? ¿Corresponde usarla como predictora tal cual está?

In [ ]:
print(sorted(df["modelo_equipo"].unique()))

**Respuesta:**

1.
2.

## 5. Histograma y densidad: la forma depende de una decisión

El ancho de barra no es un detalle de presentación: **decide qué se ve**. La regla de
Freedman–Diaconis lo fija a partir del IQR, así que es robusta:

$$h_{\text{FD}} = 2\,\frac{\text{IQR}}{n^{1/3}}$$

y la cantidad de barras es el rango dividido por ese ancho.

In [ ]:
def ancho_freedman_diaconis(x: np.ndarray) -> float:
    """Ancho de barra de Freedman-Diaconis.

    Parámetros
    ----------
    x : np.ndarray de forma (n,), sin NaN.

    Devuelve
    --------
    float : 2 * IQR / n**(1/3).
    """
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN
temp = limpio["temperatura_c"].dropna().to_numpy()
h = ancho_freedman_diaconis(temp)
n_barras = int(np.ceil((temp.max() - temp.min()) / h))
print(f"h_FD = {h:.3f} °C  ->  {n_barras} barras")

esperado = np.histogram_bin_edges(temp, bins="fd")
assert abs(len(esperado) - 1 - n_barras) <= 1, "no coincide con la regla 'fd' de NumPy"
print("Correcto: coincide con np.histogram_bin_edges(..., bins='fd')")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, bins, titulo in zip(
    axes, [5, n_barras, 200],
    ["5 barras: se pierde la estructura",
     f"{n_barras} barras (Freedman-Diaconis)",
     "200 barras: ruido de muestreo"],
):
    ax.hist(temp, bins=bins, color="#50ACB0", edgecolor="white", linewidth=0.3)
    ax.set_title(titulo, fontweight="bold", fontsize=10)
    ax.set_xlabel("Temperatura (°C)")
axes[0].set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

La densidad por núcleos no elimina la decisión: la traslada del ancho de barra al **ancho
de banda**. Su apariencia continua transmite una confianza que el método no garantiza.

In [ ]:
rejilla = np.linspace(temp.min(), temp.max(), 400)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(temp, bins=n_barras, density=True, color="#DDDDDD", edgecolor="white",
        linewidth=0.3, label="Histograma (FD)")
for bw, color, etiqueta in [(0.05, "#89A943", "h pequeño: ruido"),
                            (0.25, "#19326E", "h adecuado"),
                            (1.00, "#CD742A", "h grande: se pierde la estructura")]:
    densidad = stats.gaussian_kde(temp, bw_method=bw)
    ax.plot(rejilla, densidad(rejilla), color=color, linewidth=2, label=etiqueta)
ax.set_xlabel("Temperatura (°C)")
ax.set_ylabel("Densidad")
ax.set_title("El ancho de banda es la misma decisión que el ancho de barra", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Asimetría y curtosis de las cuatro continuas
for col in continuas:
    x = limpio[col].dropna()
    print(f"{col:20s} asimetría = {stats.skew(x):6.2f}   curtosis = {stats.kurtosis(x):8.2f}")

### DECIDE

A partir de los histogramas, de las densidades y de los coeficientes:

1. ¿Cuál de las cuatro variables continuas es **bimodal**? ¿Qué explicación de proceso tiene?
2. ¿Cuál es la más **asimétrica**? ¿Qué le haría a un modelo que supone normalidad?
3. La bimodalidad sugiere una acción concreta sobre los datos antes de modelar. ¿Cuál?

**Respuesta:**

1.
2.
3.

## 6. El diagrama de caja, construido desde cero

Los cinco números del diagrama de caja son $Q_1$, la mediana, $Q_3$ y los dos bigotes. El punto
que más se confunde es el bigote: **no llega al límite calculado**, sino al dato más extremo que
todavía cae dentro de él.

In [ ]:
def cinco_numeros(x: np.ndarray, k: float = 1.5) -> dict[str, float]:
    """Los cinco números del diagrama de caja, más los límites que no se dibujan.

    Parámetros
    ----------
    x : np.ndarray de forma (n,), sin NaN.
    k : multiplicador del IQR (1,5 es la convención de Tukey).

    Devuelve
    --------
    dict con las claves:
      "q1", "mediana", "q3", "iqr"       -> la caja
      "limite_inf", "limite_sup"         -> Q1 - k*IQR y Q3 + k*IQR (no se dibujan)
      "bigote_inf", "bigote_sup"         -> el dato más extremo DENTRO de cada límite
      "n_atipicos"                       -> cuántas observaciones quedan fuera
    """
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN — contra el cálculo que hace matplotlib para dibujar el boxplot
from matplotlib.cbook import boxplot_stats

vib = limpio["vibracion_mm_s"].dropna().to_numpy()
mio = cinco_numeros(vib)
ref = boxplot_stats(vib)[0]

assert np.isclose(mio["mediana"], ref["med"]), "mediana"
assert np.isclose(mio["q1"], ref["q1"]) and np.isclose(mio["q3"], ref["q3"]), "cuartiles"
assert np.isclose(mio["bigote_inf"], ref["whislo"]), "el bigote inferior no es un dato real"
assert np.isclose(mio["bigote_sup"], ref["whishi"]), "el bigote superior no es un dato real"
assert mio["n_atipicos"] == len(ref["fliers"]), "cantidad de atípicos"

for clave, valor in mio.items():
    print(f"  {clave:12s} {valor:10.3f}")
print("\nCorrecto: coincide con matplotlib.cbook.boxplot_stats")

La distancia entre el límite calculado y el bigote efectivo es lo que hace que los bigotes
no sean simétricos ni predecibles: su largo lo fija un dato, no una fórmula.

In [ ]:
print(f"límite superior calculado : {mio['limite_sup']:.3f}")
print(f"bigote superior dibujado  : {mio['bigote_sup']:.3f}  (un dato real)")
print(f"observaciones fuera       : {mio['n_atipicos']} de {vib.size} "
      f"({100 * mio['n_atipicos'] / vib.size:.1f} %)")

Un diagrama de caja **nunca se lee solo**. Acompañarlo con los puntos superpuestos muestra
lo que los cinco números no pueden mostrar: cuántas observaciones hay en cada zona y si la
distribución tiene una o dos modas.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
rng = np.random.default_rng(20260806)

for ax, col in zip(axes.ravel(), continuas):
    x = limpio[col].dropna().to_numpy()
    cinco = cinco_numeros(x)
    fuera = (x < cinco["limite_inf"]) | (x > cinco["limite_sup"])

    ax.boxplot(x, orientation="horizontal", widths=0.5, showfliers=False)
    ruido = rng.normal(1.0, 0.06, size=x.size)
    ax.scatter(x[~fuera], ruido[~fuera], s=6, alpha=0.15, color="#19326E",
               label="dentro de los bigotes")
    ax.scatter(x[fuera], ruido[fuera], s=14, alpha=0.7, color="#CD742A",
               label=f"fuera: {int(fuera.sum())}")
    ax.set_yticks([])
    ax.set_xlabel(col)
    ax.set_title(f"{col} — mediana {cinco['mediana']:.2f}", fontweight="bold", fontsize=10)
    ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

### Para analizar

1. En `temperatura_c`, ¿la mediana cae en una zona densa de observaciones o en un valle entre
   dos grupos? ¿Qué implica eso para un informe que reporte «la temperatura mediana»?
2. `presion_bar` y `consumo_kwh` marcan pocos puntos fuera; `vibracion_mm_s` marca muchos. ¿La
   diferencia es de calidad de los datos o de forma de la distribución?

**Respuesta:**

1.
2.

El diagrama de caja se vuelve mucho más informativo comparando grupos, porque ahí la
pregunta pasa a ser cuánto cambia la distribución **según** otra variable.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

grupos_turno = [limpio.loc[limpio["turno"] == t, "vibracion_mm_s"].dropna()
                for t in ["mañana", "tarde", "noche"]]
ax1.boxplot(grupos_turno, tick_labels=["mañana", "tarde", "noche"])
ax1.set_ylabel("Vibración (mm/s)")
ax1.set_title("Vibración por turno", fontweight="bold")

grupos_fallo = [limpio.loc[limpio["fallo"] == v, "vibracion_mm_s"].dropna() for v in [0, 1]]
ax2.boxplot(grupos_fallo, tick_labels=["sin fallo", "con fallo"])
ax2.set_ylabel("Vibración (mm/s)")
ax2.set_title("Vibración según el resultado", fontweight="bold")

plt.tight_layout()
plt.show()

print(limpio.groupby("fallo")["vibracion_mm_s"].describe().round(2).to_string())

### Para analizar

La vibración alta pertenece mayoritariamente a las máquinas que fallaron.

1. Si se «limpiaran los atípicos» de `vibracion_mm_s` antes de modelar, ¿qué se estaría
   eliminando?
2. ¿En qué se diferencia ese caso del de los 999 en `temperatura_c`?

**Respuesta:**

1.
2.

## 7. Criterios de atípicos: qué marca cada uno

Los dos criterios univariados de la clase se diferencian **solo** en con qué miden el centro y
con qué miden la escala: la regla del IQR usa cuartiles, el z-score usa media y desvío.

In [ ]:
def marcar_iqr(x: np.ndarray, k: float = 1.5) -> np.ndarray:
    """Marca atípicos con la regla del diagrama de caja.

    Parámetros
    ----------
    x : np.ndarray de forma (n,), sin NaN.
    k : multiplicador del IQR.

    Devuelve
    --------
    np.ndarray booleano de forma (n,): True donde la observación queda fuera de los límites.
    """
    raise NotImplementedError()


def marcar_zscore(x: np.ndarray, umbral: float = 3.0) -> np.ndarray:
    """Marca atípicos con el z-score clásico.

    Parámetros
    ----------
    x : np.ndarray de forma (n,), sin NaN.
    umbral : valor absoluto de z por encima del cual se marca la observación.

    Devuelve
    --------
    np.ndarray booleano de forma (n,).
    """
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN
prueba = np.array([10.0, 11, 12, 13, 14, 15, 16, 17, 18, 100])
assert marcar_iqr(prueba)[-1], "el valor 100 debe quedar marcado por la regla del IQR"
assert marcar_iqr(prueba).sum() == 1, "solo el valor 100 debe quedar marcado"
assert marcar_iqr(np.arange(100.0)).sum() == 0, "una uniforme no debería marcar nada"

m_iqr, m_z = marcar_iqr(vib), marcar_zscore(vib)
print(f"vibracion_mm_s: n = {len(vib)}")
print(f"  regla 1,5·IQR marca {m_iqr.sum():4d}  ({m_iqr.mean():.1%})")
print(f"  z-score > 3   marca {m_z.sum():4d}  ({m_z.mean():.1%})")
assert m_iqr.sum() == 199, f"se esperaban 199, se obtuvieron {m_iqr.sum()}"
print("Correcto")

Bajo normalidad, la regla $1{,}5\,$IQR marca alrededor del 0,7 % de las observaciones. Acá
marca el 5,2 %: siete veces más. No hay siete veces más anomalías; lo que el criterio está
detectando es **la forma de la distribución**, porque su calibración supone simetría.

El z-score tiene un problema propio, y es más grave. Corresponde comprobarlo sobre el
ejemplo de la clase: nueve mediciones de un sensor y una décima observación aberrante.

In [ ]:
nueve = np.array([48.0, 51, 49, 52, 50, 47, 53, 49, 51])

print(f"{'décima obs.':>12} {'s':>10} {'|z| de la décima':>18} {'¿marca?':>10}")
for extra in [60.0, 500.0, 5000.0]:
    x = np.append(nueve, extra)
    z = abs((extra - x.mean()) / x.std(ddof=1))
    print(f"{extra:12.0f} {x.std(ddof=1):10.2f} {z:18.2f} {str(z > 3):>10}")

n = 10
print(f"\nCota teórica del z-score con n = {n}: (n-1)/sqrt(n) = {(n - 1) / np.sqrt(n):.3f}")
print("Con umbral 3, ninguna observación puede ser marcada: el umbral es inalcanzable.")

### Para analizar

1. Al pasar de 60 a 5000, la observación se vuelve muchísimo más extrema y su $|z|$ casi no se
   mueve. ¿Por qué?
2. ¿Qué criterio de los dos habría marcado la observación 500? Comprobarlo.

In [ ]:
# TODO: aplicar marcar_iqr al arreglo de diez valores con la décima observación en 500
#       e informar cuántas y cuáles observaciones quedan marcadas.
raise NotImplementedError()

**Respuesta:**

1.
2.

## 8. Relaciones entre variables: cuando Pearson y Spearman no coinciden

Pearson mide asociación **lineal** y no resiste valores extremos. Spearman es Pearson sobre los
rangos: mide asociación **monótona** y sí los resiste. Cuando los dos difieren mucho, la
diferencia misma es el hallazgo.

In [ ]:
def divergencia_pearson_spearman(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """Ordena los pares de variables según cuánto difieren Pearson y Spearman.

    Parámetros
    ----------
    df : DataFrame con las variables.
    cols : columnas numéricas a considerar.

    Devuelve
    --------
    pd.DataFrame con columnas ["var_1", "var_2", "pearson", "spearman", "divergencia"],
    ordenado por "divergencia" de mayor a menor. Una fila por par, sin repetir A-B y B-A.
    """
    raise NotImplementedError()

In [ ]:
# VERIFICACIÓN
ranking = divergencia_pearson_spearman(limpio, numericas)
print(ranking.head(5).round(3).to_string(index=False))

peor = ranking.iloc[0]
assert {peor.var_1, peor.var_2} == {"horas_operacion", "antiguedad_meses"}, \
    "el par más divergente debería ser horas_operacion / antiguedad_meses"
print(f"\nCorrecto: {peor.var_1} vs {peor.var_2} -> "
      f"Pearson {peor.pearson:.3f} contra Spearman {peor.spearman:.3f}")

In [ ]:
# El gráfico que explica la divergencia
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.scatter(limpio["antiguedad_meses"], limpio["horas_operacion"], s=10, alpha=0.4,
            color="#19326E")
ax1.set_xlabel("Antigüedad (meses)"); ax1.set_ylabel("Horas de operación")
ax1.set_title("Escala original", fontweight="bold")

mask = limpio["horas_operacion"] < 100_000
ax2.scatter(limpio.loc[mask, "antiguedad_meses"], limpio.loc[mask, "horas_operacion"],
            s=10, alpha=0.4, color="#19326E")
ax2.set_xlabel("Antigüedad (meses)"); ax2.set_ylabel("Horas de operación")
ax2.set_title("Excluyendo los valores extremos", fontweight="bold")
plt.tight_layout(); plt.show()

print(f"Registros con horas_operacion > 100.000: {(~mask).sum()}")

### Para analizar

1. ¿Cuántos registros producen la divergencia y qué porcentaje del total representan?
2. ¿Qué defecto concreto tienen esos registros? Verificar la hipótesis con una cuenta.
3. ¿Cuál de los dos coeficientes describe mejor la relación real entre antigüedad y horas?

**Respuesta:**

1.
2.
3.

## 9. Datos faltantes: el mecanismo decide el tratamiento

Contar faltantes por columna es el primer paso, no el último. Lo que decide el tratamiento es
**por qué** faltan.

In [ ]:
faltan = limpio.isna().sum()
print("Faltantes por variable:")
print(faltan[faltan > 0].to_string())
print()

for col in ["consumo_kwh", "vibracion_mm_s"]:
    print(f"--- Tasa de faltantes de {col} por grupo ---")
    for grupo in ["planta", "turno"]:
        tasa = limpio.assign(f=limpio[col].isna()).groupby(grupo)["f"].mean() * 100
        print(f"  por {grupo}: " + "  ".join(f"{k}={v:.1f}%" for k, v in tasa.items()))
    print()

obs = limpio["temperatura_c"].dropna()
print("--- temperatura_c (ya sin el código 999) ---")
print(f"  máximo observado: {obs.max():.1f} °C")
print(f"  faltantes: {int(limpio['temperatura_c'].isna().sum())}")

In [ ]:
print("Presencia de costo_reparacion_usd según el resultado:")
print(pd.crosstab(limpio["fallo"], limpio["costo_reparacion_usd"].notna(),
                  rownames=["fallo"], colnames=["costo informado"]))

### DECIDE

1. Clasificar el mecanismo de `consumo_kwh`, `vibracion_mm_s` y `temperatura_c` en MCAR, MAR o
   MNAR, citando la evidencia de las celdas anteriores.
2. `costo_reparacion_usd` falta en el 90 % de los registros y está presente **exactamente** en
   los que tienen `fallo = 1`. ¿Es una variable con faltantes? ¿Puede usarse como predictora?
3. ¿Cuál de los tres mecanismos **no** se puede diagnosticar con los datos disponibles, y qué
   haría falta para detectarlo?

**Respuesta:**

1.
2.
3.

## 10. Informe de hallazgos

Es el entregable. Una fila por hallazgo, con las cuatro columnas completas. Un hallazgo sin
evidencia numérica no es un hallazgo, y una decisión sin justificación no es auditable.

### Hallazgos

| Evidencia | Criterio | Decisión | Justificación |
|---|---|---|---|
| *ej.: `temperatura_c` toma el valor 999,0 en 31 registros* | *repetición exacta + fuera de la valla 3·IQR* | *reemplazar por faltante y registrar* | *no es una temperatura posible; conservarlo multiplica el desvío por 8,7* |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |
|  |  |  |  |

### Tres consecuencias para el modelado

Cada una debe seguirse de un hallazgo de la tabla.

1.
2.
3.

## Cierre

- Un resumen numérico comprime, y al comprimir esconde. Cada estadístico de este laboratorio
  tuvo que mirarse junto a un gráfico para significar algo.
- Los criterios de atípicos no son neutrales: cada uno supone una forma. Cuando el supuesto no
  se cumple, el criterio detecta la forma de la distribución, no anomalías.
- Un atípico no es un error: es una observación que hay que explicar. En este conjunto conviven
  las cuatro causas, y cada una pide un tratamiento distinto.

Lo que no se haya completado durante la clase se termina fuera de ella. El informe de la sección
10 se entrega completo, con las cuatro columnas y las tres consecuencias.

## Para seguir explorando

- Repetir el análisis de la sección 6 sobre `temperatura_c` separando por régimen de operación
  (por encima y por debajo de 75 °C). ¿Qué pasa con la cantidad de atípicos marcados?
- Comparar la cantidad de atípicos que marca la regla $1{,}5\,$IQR sobre `vibracion_mm_s` antes y
  después de aplicar una transformación logarítmica. Interpretar el cambio.